# Sugestão de playlists por ambiente

Este notebook recomenda uma sequência de músicas a partir do ambiente desejado. Os atributos acústicos são comparados com um perfil de referência e a sequência é reorganizada para favorecer tonalidades positivas, próximas e transições harmônicas.

O filtro de conteúdo explícito é opcional e fica ativado por padrão.

## 1. Ambiente da playlist

Os perfis abaixo são pontos de partida. Eles podem ser ajustados na célula seguinte conforme a ocasião e o público. Cada atributo varia de 0 a 1.

In [ ]:
from pathlib import Path
from IPython.display import HTML
import numpy as np
import pandas as pd

base_dir = Path.cwd().parent
cleaned_files = sorted((base_dir / 'dataset').glob('dataset_cleaned_*.csv'))
if not cleaned_files:
    raise FileNotFoundError('Nenhum dataset limpo foi encontrado em dataset/.')

csv_path = cleaned_files[-1]
tracks = pd.read_csv(csv_path)
clustered_path = base_dir / 'dataset' / 'tracks_with_clusters.csv'
centroids_path = base_dir / 'dataset' / 'cluster_centroids.csv'
if clustered_path.exists() and 'track_id' in tracks.columns:
    clustered_tracks = pd.read_csv(clustered_path, usecols=['track_id', 'cluster'])
    tracks = tracks.merge(clustered_tracks, on='track_id', how='left')
if centroids_path.exists():
    cluster_centroids = pd.read_csv(centroids_path, index_col=0)
else:
    cluster_centroids = pd.DataFrame()

print(f'Dataset utilizado: {csv_path.name}')
print(f'Faixas disponíveis: {len(tracks):,}'.replace(',', '.'))
print(f'Clusters carregados: {tracks["cluster"].nunique() if "cluster" in tracks.columns else 0}')
print('Colunas musicais encontradas:', [column for column in ['danceability', 'energy', 'instrumentalness', 'liveness', 'valence', 'key', 'mode', 'explicit'] if column in tracks.columns])

Dataset utilizado: dataset_cleaned_20260902_094950.csv
Faixas disponíveis: 79.178
Colunas musicais encontradas: ['danceability', 'energy', 'instrumentalness', 'liveness', 'valence', 'key', 'mode', 'explicit']


## 2. Filtragem, clusters e pontuação por ambiente

A playlist combina a distância média aos cinco atributos musicais com o contexto de gênero escolhido. Depois dos filtros, as faixas são selecionadas entre os clusters disponíveis, priorizando os menos representados para aumentar a diversidade sonora. Se o contexto escolhido deixar apenas um cluster disponível, o algoritmo mantém esse cluster sem forçar faixas incompatíveis. A popularidade serve como critério secundário para desempate.

In [ ]:
genre_taxonomy = {
    'Rock, Metal & Punk': {
        'Alta Positividade': ['rock-n-roll', 'rockabilly'],
        'Positividade Média': ['alt-rock', 'grunge', 'hard-rock', 'psych-rock', 'punk-rock', 'rock'],
        'Baixa Positividade': ['black-metal', 'death-metal', 'grindcore', 'hardcore', 'heavy-metal', 'industrial', 'metal', 'metalcore', 'punk']
    },
    'Eletrônica & Dance': {
        'Alta Positividade': ['chicago-house', 'club', 'dance', 'disco', 'edm', 'electro', 'house', 'synth-pop'],
        'Positividade Média': ['breakbeat', 'deep-house', 'detroit-techno', 'drum-and-bass', 'electronic', 'garage', 'idm', 'minimal-techno', 'progressive-house', 'techno', 'trance', 'trip-hop'],
        'Baixa Positividade': ['dubstep', 'hardstyle']
    },
    'Pop, Indie & Alternativo': {
        'Alta Positividade': ['indie-pop', 'pop', 'power-pop'],
        'Positividade Média': ['alternative', 'indie'],
        'Baixa Positividade': ['emo', 'goth']
    },
    'Hip-Hop, R&B & Soul': {
        'Alta Positividade': ['funk', 'groove'],
        'Positividade Média': ['hip-hop', 'r-n-b', 'soul']
    },
    'Música Brasileira': {
        'Alta Positividade': ['brazil', 'forro', 'pagode', 'samba'],
        'Positividade Média': ['mpb', 'sertanejo']
    },
    'Música Latina': {
        'Alta Positividade': ['latin', 'latino', 'reggaeton', 'salsa'],
        'Positividade Média': ['spanish', 'tango']
    },
    'Folk, Country & Acústico': {
        'Alta Positividade': ['bluegrass', 'honky-tonk'],
        'Positividade Média': ['acoustic', 'country', 'folk', 'guitar', 'singer-songwriter', 'songwriter']
    },
    'Jazz, Blues & Clássica': {
        'Positividade Média': ['classical', 'jazz', 'opera', 'piano'],
        'Baixa Positividade': ['blues']
    },
    'Reggae, Dub & Afrobeat': {
        'Alta Positividade': ['afrobeat', 'dancehall', 'ska'],
        'Positividade Média': ['dub', 'reggae']
    },
    'Música Asiática': {
        'Alta Positividade': ['anime', 'cantopop', 'j-dance', 'j-idol', 'j-pop', 'k-pop', 'mandopop'],
        'Positividade Média': ['j-rock']
    },
    'Músicas do Mundo & Regionais': {
        'Positividade Média': ['british', 'french', 'german', 'indian', 'iranian', 'malay', 'swedish', 'turkish', 'world-music']
    },
    'Climas, Moods & Atividades': {
        'Alta Positividade': ['happy', 'party'],
        'Positividade Média': ['ambient', 'chill', 'new-age'],
        'Baixa Positividade': ['sad', 'sleep']
    },
    'Infantil, Trilhas Sonoras & Temáticas': {
        'Alta Positividade': ['children', 'comedy', 'disney', 'kids', 'show-tunes'],
        'Positividade Média': ['gospel', 'pop-film']
    }
}


def filter_candidates(tracks, profile, feature_ranges, avoid_explicit, popularity_range, allowed_genres):
    feature_columns = ['danceability', 'instrumentalness', 'valence', 'energy', 'liveness']
    required_columns = feature_columns + ['key', 'mode', 'track_name', 'artists', 'track_genre', 'explicit', 'popularity']
    missing_columns = sorted(set(required_columns) - set(tracks.columns))
    if missing_columns:
        raise ValueError(f'Colunas obrigatórias ausentes: {missing_columns}')

    candidates = tracks.copy()
    if avoid_explicit:
        candidates = candidates.loc[~candidates['explicit'].astype(bool)].copy()

    for column in feature_columns + ['key', 'mode', 'popularity']:
        candidates[column] = pd.to_numeric(candidates[column], errors='coerce')
    candidates['track_genre'] = candidates['track_genre'].astype(str).str.lower().str.strip()
    candidates = candidates.dropna(subset=required_columns).copy()
    candidates = candidates.loc[candidates['key'].between(0, 11)].copy()
    candidates = candidates.loc[candidates['mode'].isin([0, 1])].copy()
    for feature, (minimum, maximum) in feature_ranges.items():
        candidates = candidates.loc[candidates[feature].between(minimum, maximum)].copy()
    min_popularity, max_popularity = popularity_range
    candidates = candidates.loc[candidates['popularity'].between(min_popularity, max_popularity)].copy()
    if allowed_genres:
        candidates = candidates.loc[candidates['track_genre'].isin(allowed_genres)].copy()

    target = pd.Series({feature: sum(bounds) / 2 for feature, bounds in feature_ranges.items()})
    candidates['environment_distance'] = candidates[feature_columns].sub(target[feature_columns], axis='columns').abs().mean(axis=1)
    candidates['fit_score'] = 1 - candidates['environment_distance']
    return candidates.sort_values(['environment_distance', 'popularity'], ascending=[True, False])

## 3. Sequenciamento harmônico

`key` segue a Pitch Class notation: `0` = C até `11` = B. O valor `-1` indica tonalidade não detectada e é removido antes do sequenciamento. `mode` vale `1` para maior e `0` para menor. O resultado também exibe a notação Camelot (`1A` a `12A` para menores e `1B` a `12B` para maiores). A distância circular evita saltos grandes; a distância 6 é o trítono e recebe uma penalidade máxima.

In [ ]:
def key_distance(first_key, second_key):
    distance = abs(int(first_key) - int(second_key)) % 12
    return min(distance, 12 - distance)


def transition_cost(previous, candidate):
    distance = key_distance(previous['key'], candidate['key'])
    mode_change = int(previous['mode'] != candidate['mode'])
    opposite_key = int(distance == 6)
    return distance * 0.08 + mode_change * 0.18 + opposite_key * 10


def build_playlist(candidates, size, positive_tonality=True, seed=None):
    rng = np.random.default_rng(seed)
    remaining = candidates.copy()
    if positive_tonality:
        major = remaining.loc[remaining['mode'] == 1]
        if not major.empty:
            remaining = major

    if remaining.empty:
        return remaining

    first_pool = remaining.nlargest(min(8, len(remaining)), 'fit_score')
    if 'cluster' in first_pool.columns and first_pool['cluster'].notna().any():
        first_pool = first_pool.drop_duplicates('cluster')
    first_index = rng.choice(first_pool.index.to_numpy())
    selected = [first_index]
    remaining = remaining.drop(index=first_index)
    cluster_counts = candidates.loc[selected, 'cluster'].value_counts().to_dict() if 'cluster' in candidates.columns else {}

    while len(selected) < min(size, len(candidates)) and not remaining.empty:
        previous = candidates.loc[selected[-1]]
        transition_distances = remaining.apply(
            lambda candidate: key_distance(previous['key'], candidate['key']), axis=1
        )
        compatible = remaining.loc[transition_distances != 6]
        if compatible.empty:
            break
        transition_scores = compatible.apply(
            lambda candidate: transition_cost(previous, candidate), axis=1
        )
        ranking = compatible['fit_score'] - transition_scores
        candidate_pool = ranking.nlargest(min(24, len(ranking))).index
        if 'cluster' in compatible.columns and compatible.loc[candidate_pool, 'cluster'].notna().any():
            pool = compatible.loc[candidate_pool].copy()
            pool['_cluster_count'] = pool['cluster'].map(cluster_counts).fillna(0)
            least_represented = pool['_cluster_count'].min()
            candidate_pool = pool.index[pool['_cluster_count'] == least_represented]
        next_index = rng.choice(candidate_pool.to_numpy())
        selected.append(next_index)
        remaining = remaining.drop(index=next_index)
        if 'cluster' in candidates.columns:
            cluster = candidates.loc[next_index, 'cluster']
            cluster_counts[cluster] = cluster_counts.get(cluster, 0) + 1

    return candidates.loc[selected]

In [ ]:
def camelot_key(key, mode):
    major_keys = {0: '8B', 1: '3B', 2: '10B', 3: '5B', 4: '12B', 5: '7B', 6: '2B', 7: '9B', 8: '4B', 9: '11B', 10: '6B', 11: '1B'}
    minor_keys = {0: '5A', 1: '12A', 2: '7A', 3: '2A', 4: '9A', 5: '4A', 6: '11A', 7: '6A', 8: '1A', 9: '8A', 10: '3A', 11: '10A'}
    key_map = major_keys if int(mode) == 1 else minor_keys
    return key_map.get(int(key))


def generate_playlist(environment, profile, feature_ranges, playlist_size, avoid_explicit, prefer_positive_tonality, selected_genre_groups, selected_positivity_levels, allowed_genres, popularity_range, random_seed=None):
    global candidates, playlist
    candidates = filter_candidates(tracks, profile, feature_ranges, avoid_explicit, popularity_range, allowed_genres)
    playlist = build_playlist(candidates, playlist_size, prefer_positive_tonality, random_seed)
    if playlist.empty:
        raise ValueError('Nenhuma faixa corresponde aos filtros selecionados.')

    playlist.insert(0, 'position', range(1, len(playlist) + 1))
    playlist['key_distance_from_previous'] = [
        np.nan,
        *[key_distance(playlist.iloc[index - 1]['key'], playlist.iloc[index]['key']) for index in range(1, len(playlist))]
    ]
    playlist['key_camelot'] = [camelot_key(key, mode) for key, mode in zip(playlist['key'], playlist['mode'])]
    transition_distances = playlist['key_distance_from_previous'].dropna()
    tritone_transitions = int((transition_distances == 6).sum())
    explicit_tracks = int(playlist['explicit'].astype(bool).sum())
    assert tritone_transitions == 0, 'A playlist contém uma transição de tonalidades opostas.'
    if avoid_explicit:
        assert explicit_tracks == 0, 'O filtro explícito está ativo, mas uma faixa explícita foi selecionada.'

    output_columns = ['position', 'track_name', 'artists', 'track_genre', 'cluster', 'popularity', 'fit_score', 'key_camelot', 'mode', 'key_distance_from_previous', 'explicit']
    result = playlist[[column for column in output_columns if column in playlist.columns]]
    display(HTML(f'<div style="width: 100%; max-width: 100%; overflow-x: auto;"><div style="min-width: 1200px;">{result.to_html(index=False, escape=True)}</div></div>'))
    print(f'Playlist gerada: {len(playlist)} faixas')
    print(f'Clusters utilizados: {playlist["cluster"].nunique() if "cluster" in playlist.columns else "indisponíveis"}')
    print(f'Gêneros selecionados: {len(selected_genre_groups)} grupos / {len(selected_positivity_levels)} níveis')
    print(f'Faixa de popularidade aplicada: {popularity_range[0]} a {popularity_range[1]}')
    print(f'Semente aleatória: {random_seed if random_seed is not None else "aleatória"}')
    print(f'Transições de trítono: {tritone_transitions}')
    print(f'Faixas explícitas: {explicit_tracks}')

    playlist.to_csv(base_dir / 'dataset' / f'playlist_{environment}.csv', index=False)
    print(f'Arquivo salvo em: dataset/playlist_{environment}.csv')
    return playlist

## Interpretação

A playlist não representa uma verdade musical universal: os perfis são heurísticas e podem ser calibrados com avaliações reais dos ouvintes. O sequenciamento evita o trítono, mas não substitui uma análise harmônica detalhada de introduções, BPM ou estrutura das músicas.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

environment_profiles = {
    'festa': {'danceability': 0.85, 'instrumentalness': 0.10, 'valence': 0.80, 'energy': 0.85, 'liveness': 0.25},
    'treino': {'danceability': 0.75, 'instrumentalness': 0.05, 'valence': 0.65, 'energy': 0.90, 'liveness': 0.20},
    'estudo': {'danceability': 0.35, 'instrumentalness': 0.55, 'valence': 0.55, 'energy': 0.35, 'liveness': 0.10},
    'jantar': {'danceability': 0.55, 'instrumentalness': 0.30, 'valence': 0.70, 'energy': 0.45, 'liveness': 0.15},
    'relaxamento': {'danceability': 0.25, 'instrumentalness': 0.65, 'valence': 0.60, 'energy': 0.20, 'liveness': 0.08}
}

environment_genre_profiles = {
    'festa': {'groups': ['Eletrônica & Dance', 'Pop, Indie & Alternativo', 'Hip-Hop, R&B & Soul', 'Música Brasileira', 'Música Latina', 'Reggae, Dub & Afrobeat'], 'levels': ['Alta Positividade', 'Positividade Média']},
    'treino': {'groups': ['Eletrônica & Dance', 'Rock, Metal & Punk', 'Hip-Hop, R&B & Soul', 'Reggae, Dub & Afrobeat'], 'levels': ['Alta Positividade', 'Positividade Média']},
    'estudo': {'groups': ['Jazz, Blues & Clássica', 'Folk, Country & Acústico', 'Climas, Moods & Atividades'], 'levels': ['Positividade Média']},
    'jantar': {'groups': ['Jazz, Blues & Clássica', 'Música Brasileira', 'Música Latina', 'Folk, Country & Acústico', 'Pop, Indie & Alternativo'], 'levels': ['Alta Positividade', 'Positividade Média']},
    'relaxamento': {'groups': ['Climas, Moods & Atividades', 'Jazz, Blues & Clássica', 'Folk, Country & Acústico', 'Eletrônica & Dance'], 'levels': ['Positividade Média']}
}

style = {'description_width': 'initial'}
environment_widget = widgets.Dropdown(options=list(environment_profiles), value='festa', description='Ambiente:', style=style)
feature_widgets = {
    feature: widgets.FloatRangeSlider(
        value=(max(0, value - 0.10), min(1, value + 0.10)), min=0, max=1, step=0.05,
        description=f'{feature}:', continuous_update=False, readout_format='.2f', style=style,
        layout=widgets.Layout(width='500px')
    )
    for feature, value in environment_profiles['festa'].items()
}
group_widget = widgets.SelectMultiple(options=list(genre_taxonomy), description='Grupos:', rows=7, style=style, layout=widgets.Layout(width='500px'))
level_widget = widgets.SelectMultiple(options=['Alta Positividade', 'Positividade Média', 'Baixa Positividade'], description='Positividade:', rows=3, style=style)
popularity_widget = widgets.IntRangeSlider(value=(0, 100), min=0, max=100, step=1, description='Popularidade:', continuous_update=False, style=style, layout=widgets.Layout(width='500px'))
seed_widget = widgets.IntText(value=0, min=0, description='Semente (0 = aleatória):', style=style)
playlist_size_widget = widgets.IntSlider(value=20, min=5, max=50, step=1, description='Faixas:', continuous_update=False, style=style)
avoid_explicit_widget = widgets.Checkbox(value=True, description='Evitar conteúdo explícito')
positive_tonality_widget = widgets.Checkbox(value=True, description='Priorizar tonalidades maiores')
apply_button = widgets.Button(description='Aplicar parâmetros', button_style='primary', icon='check')
output = widgets.Output(layout=widgets.Layout(width='100%', max_width='100%', overflow_x='auto'))


def load_profile(change=None):
    profile = environment_profiles[environment_widget.value]
    genre_profile = environment_genre_profiles[environment_widget.value]
    for feature, widget in feature_widgets.items():
        widget.value = (max(0, profile[feature] - 0.10), min(1, profile[feature] + 0.10))
    group_widget.value = tuple(genre_profile['groups'])
    level_widget.value = tuple(genre_profile['levels'])


def apply_parameters(button):
    global environment, profile, feature_ranges, playlist_size, avoid_explicit, prefer_positive_tonality
    global selected_genre_groups, selected_positivity_levels, allowed_genres, popularity_range, random_seed
    environment = environment_widget.value
    feature_ranges = {feature: tuple(widget.value) for feature, widget in feature_widgets.items()}
    profile = {feature: sum(bounds) / 2 for feature, bounds in feature_ranges.items()}
    playlist_size = playlist_size_widget.value
    avoid_explicit = avoid_explicit_widget.value
    prefer_positive_tonality = positive_tonality_widget.value
    selected_genre_groups = list(group_widget.value)
    selected_positivity_levels = list(level_widget.value)
    allowed_genres = sorted({genre for group in selected_genre_groups for level in selected_positivity_levels for genre in genre_taxonomy[group].get(level, [])})
    popularity_range = tuple(popularity_widget.value)
    random_seed = seed_widget.value if seed_widget.value > 0 else None
    with output:
        clear_output(wait=True)
        try:
            generate_playlist(environment, profile, feature_ranges, playlist_size, avoid_explicit, prefer_positive_tonality, selected_genre_groups, selected_positivity_levels, allowed_genres, popularity_range, random_seed)
        except ValueError as error:
            print(f'Não foi possível gerar a playlist: {error}')


environment_widget.observe(load_profile, names='value')
apply_button.on_click(apply_parameters)
load_profile()
display(widgets.VBox([environment_widget, widgets.HTML('<b>Defina o intervalo de cada atributo (0 a 1)</b>'), *feature_widgets.values(), group_widget, level_widget, popularity_widget, seed_widget, playlist_size_widget, avoid_explicit_widget, positive_tonality_widget, apply_button, output]))
apply_parameters(None)